# Step Dataset Analysis — Fundamental Questions

Data: `simulation/results/step_dataset/qwen3_14b/` (10 tasks per benchmark, captured with EAGLE-3 backbone at S=8/K=16 + suffix decoding at no F/T filter, oracle force-1 mode).

**Priorities covered**:
- **P1.1** Accept depth survival — *how deep can we accept?*
- **P1.2** Cost vs reward — *is suffix worth its verification cost?*
- **P2.3** Anchor-depth × suffix gain — *where on the backbone is suffix useful?*
- **P2.5** EAGLE calibration — *does `p_eagle` match observed accept rate?*
- **P3.4** Feature lift — *what predicts suffix acceptance?*
- **Workload comparison** runs through every section as a facet.

## 0. Setup

In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_ROOT = Path("/workspace/simulation/results/step_dataset/qwen3_14b")
BENCHES = ["bfcl_v4", "specbench", "swebench_verified"]
PALETTE = dict(zip(BENCHES, sns.color_palette("Set2", 3)))

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
def load_jsonl_gz(path_glob_pattern):
    """Read a sharded set of .jsonl.gz files into a DataFrame."""
    rows = []
    for p in sorted(DATA_ROOT.glob(path_glob_pattern)):
        if p.stat().st_size == 0:
            continue
        with gzip.open(p, "rb") as f:
            for line in f:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def load_tsv(path):
    return pd.read_csv(path, sep="\t") if path.exists() else pd.DataFrame()


def load_for_bench(bench, families=("01_step_raw", "06_cost_raw")):
    out = {}
    for fam in families:
        out[fam] = load_jsonl_gz(f"{bench}/{fam}.shard*.jsonl.gz")
        out[fam]["benchmark"] = bench
    return out


# Load run index (small)
run_index = load_tsv(DATA_ROOT / "00_run_index.tsv")
run_index

In [ ]:
# Load step_raw + cost_raw for all 3 benchmarks
step_dfs = []
cost_dfs = []
for b in BENCHES:
    d = load_for_bench(b, families=("01_step_raw", "06_cost_raw"))
    step_dfs.append(d["01_step_raw"])
    cost_dfs.append(d["06_cost_raw"])
step_df = pd.concat(step_dfs, ignore_index=True)
cost_df = pd.concat(cost_dfs, ignore_index=True)
print(f"step_df: {len(step_df):,} rows across {step_df['benchmark'].nunique()} benchmarks")
print(f"cost_df: {len(cost_df):,} rows")
step_df.groupby("benchmark").agg(
    n_steps=("step_id", "count"),
    n_tasks=("sample_id", "nunique"),
    mean_backbone=("backbone_size", "mean"),
    mean_extended=("extended_size", "mean"),
    mean_acc_len=("accepted_len_via_max_tree", "mean"),
).round(2)

## P1.1 — Accept Depth Survival

**Question**: How deep does the accepted path typically reach? Does adding suffix push beyond the EAGLE backbone limit?

We have two depth-frontiers per step:
- `accepted_max_depth_backbone` — limited by EAGLE backbone (max S=8).
- `accepted_max_depth_extended` — can exceed 8 when suffix carries the accept further.

Plot survival curves: P[accepted_depth >= d] for each benchmark.

In [ ]:
def survival_curve(series, max_d=24):
    n = len(series)
    return pd.Series(
        {d: (series >= d).mean() for d in range(1, max_d + 1)})


fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for b in BENCHES:
    sub = step_df[step_df["benchmark"] == b]
    bb = survival_curve(sub["accepted_max_depth_backbone"])
    ext = survival_curve(sub["accepted_max_depth_extended"])
    axes[0].plot(bb.index, bb.values, label=b, color=PALETTE[b], marker="o")
    axes[1].plot(ext.index, ext.values, label=b, color=PALETTE[b], marker="o")
axes[0].set_title("Backbone-only accept (EAGLE)")
axes[1].set_title("Extended accept (EAGLE + suffix)")
for ax in axes:
    ax.set_xlabel("depth d")
    ax.set_ylabel("P[accepted depth ≥ d]")
    ax.set_yscale("log")
    ax.legend()
    ax.axvline(8, ls="--", color="grey", alpha=0.5, label="backbone limit S=8")
plt.tight_layout()
plt.show()

In [ ]:
# Conditional accept rate: P[depth >= d | depth >= d-1]
rows = []
for b in BENCHES:
    sub = step_df[step_df["benchmark"] == b]
    for col, prop in [("accepted_max_depth_backbone", "backbone"),
                       ("accepted_max_depth_extended", "extended")]:
        for d in range(1, 17):
            ge_d = (sub[col] >= d).sum()
            ge_dm1 = (sub[col] >= d - 1).sum()
            cond = ge_d / ge_dm1 if ge_dm1 > 0 else float("nan")
            rows.append({"benchmark": b, "proposer": prop, "depth": d,
                         "conditional_accept": cond})
cond_df = pd.DataFrame(rows)

g = sns.relplot(cond_df, x="depth", y="conditional_accept",
                hue="benchmark", col="proposer", kind="line",
                marker="o", palette=PALETTE, height=4, aspect=1.2)
g.set_axis_labels("depth d", "P[accept depth ≥ d | accept ≥ d-1]")
g.fig.suptitle("Conditional accept rate by depth", y=1.02)
plt.show()

**Reading**: a high horizontal line means "once you've reached this depth, the next layer accepts too". A steep drop means EAGLE quickly loses confidence beyond that depth.

## P1.2 — Cost vs Reward (is suffix worth the verification cost?)

Per step we have two scenarios:
- **backbone_only**: target verifies only the EAGLE backbone tree (~1808 nodes). `target_latency_backbone_only_ms`.
- **max_tree**: target verifies the full extended tree (backbone + every suffix graft, ~5-7k nodes). `target_latency_max_tree_ms`.

Metric: `accepted_len / total_latency_ms` (tokens per ms). Higher is better.

If backbone-only beats max-tree, **suffix is not worth verifying everything** — selection becomes critical. If max-tree wins, suffix tokens land enough to justify the cost.

In [ ]:
c = cost_df.copy()
c["tok_per_ms_max"] = c["accepted_len_via_max_tree"] / c["total_latency_max_tree_ms"]
c["tok_per_ms_bb"] = c["accepted_len_via_max_tree"] / c["total_latency_backbone_only_ms"]

summary = c.groupby("benchmark").agg(
    mean_acc_len=("accepted_len_via_max_tree", "mean"),
    mean_lat_max=("total_latency_max_tree_ms", "mean"),
    mean_lat_bb=("total_latency_backbone_only_ms", "mean"),
    tok_per_ms_max=("tok_per_ms_max", "mean"),
    tok_per_ms_bb=("tok_per_ms_bb", "mean"),
).round(4)
summary["backbone_advantage"] = (summary["tok_per_ms_bb"] /
                                  summary["tok_per_ms_max"] - 1) * 100
summary

In [ ]:
# Plot per-benchmark distribution of (max_tree − backbone_only) latency
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
diff = c.assign(
    extra_cost_ms=c["total_latency_max_tree_ms"] - c["total_latency_backbone_only_ms"]
)
sns.boxplot(diff, x="benchmark", y="extra_cost_ms",
            order=BENCHES, palette=PALETTE, ax=ax, showfliers=False)
ax.set_title("Extra verification cost from suffix grafts (per step)")
ax.set_ylabel("ms")
plt.show()

**Caveat**: this assumes accept_len is the SAME under both scenarios (= what greedy walk gives on the extended tree). In practice the backbone-only scenario could accept less (no suffix to extend the path), so the backbone-only `tok_per_ms` is an *upper bound* under the assumption that suffix never helps. The real backbone-only accept_len would be `accepted_max_depth_backbone`. Let's redo with that.

In [ ]:
# Realistic comparison: backbone-only accept = accepted_max_depth_backbone
c2 = c.merge(
    step_df[["benchmark", "sample_id", "call_idx", "step_id",
             "accepted_max_depth_backbone", "accepted_max_depth_extended"]],
    on=["benchmark", "sample_id", "call_idx", "step_id"],
)
c2["tok_per_ms_max_real"] = c2["accepted_max_depth_extended"] / c2["total_latency_max_tree_ms"]
c2["tok_per_ms_bb_real"] = c2["accepted_max_depth_backbone"] / c2["total_latency_backbone_only_ms"]

summary2 = c2.groupby("benchmark").agg(
    acc_max=("accepted_max_depth_extended", "mean"),
    acc_bb=("accepted_max_depth_backbone", "mean"),
    tok_per_ms_max=("tok_per_ms_max_real", "mean"),
    tok_per_ms_bb=("tok_per_ms_bb_real", "mean"),
).round(4)
summary2["suffix_advantage_pct"] = (
    summary2["tok_per_ms_max"] / summary2["tok_per_ms_bb"] - 1) * 100
summary2

**Interpretation**: if `suffix_advantage_pct` > 0, the extra accept from suffix justifies the extra verify cost; if < 0, max-tree is wasteful and selection is needed.

## P2.3 — Anchor depth × suffix gain

**Question**: at which EAGLE anchor depth does suffix actually contribute? Use the pre-aggregated `04_anchor_depth_summary.tsv`.

In [ ]:
ad_dfs = []
for b in BENCHES:
    p = DATA_ROOT / b / "04_anchor_depth_summary.tsv"
    ad_dfs.append(load_tsv(p))
ad_df = pd.concat(ad_dfs, ignore_index=True)
ad_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric, ylabel in zip(
    axes,
    ["anchor_hit_rate", "avg_suffix_accept_len_cond_hit",
     "avg_suffix_uncond_contribution"],
    ["P[anchor on accepted path]",
     "E[suffix accept | anchor hit]",
     "Unconditional suffix contribution"],
):
    for b in BENCHES:
        sub = ad_df[ad_df["benchmark"] == b].sort_values("anchor_depth")
        ax.plot(sub["anchor_depth"], sub[metric],
                marker="o", color=PALETTE[b], label=b)
    ax.set_xlabel("anchor_depth")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    if metric == "anchor_hit_rate":
        ax.set_yscale("log")
plt.tight_layout()
plt.show()

**Reading**:
- `anchor_hit_rate` drops exponentially with depth → deep anchors rarely on accepted path.
- `avg_suffix_accept_len_cond_hit` tells you, *if* the anchor is hit, how many extra suffix tokens we get.
- `avg_suffix_uncond_contribution` = `hit_rate × cond_accept`. This is the actual marginal value at this depth.

## P2.5 — EAGLE calibration (is `p_eagle` well calibrated?)

We compare bin-centered `anchor_path_prob_eagle` to the observed `anchor_hit` rate per bin. A perfectly calibrated model gives diagonal y=x.

We load `03_anchor_raw` lazily (it's large).

In [ ]:
def stream_anchor_calibration(bench, n_bins=20):
    """Stream-bin: avoid loading 100M rows into memory."""
    edges = np.linspace(0, 1, n_bins + 1)
    sum_p = np.zeros(n_bins); n_p = np.zeros(n_bins); n_hit = np.zeros(n_bins)
    for p in sorted((DATA_ROOT / bench).glob("03_anchor_raw.shard*.jsonl.gz")):
        if p.stat().st_size == 0:
            continue
        with gzip.open(p, "rb") as f:
            for line in f:
                r = json.loads(line)
                pp = r.get("anchor_path_prob_eagle")
                if pp is None:
                    continue
                b = min(n_bins - 1, max(0, int(pp * n_bins)))
                sum_p[b] += pp
                n_p[b] += 1
                if r.get("anchor_hit"):
                    n_hit[b] += 1
    avg_p = np.where(n_p > 0, sum_p / np.maximum(n_p, 1), np.nan)
    hit_rate = np.where(n_p > 0, n_hit / np.maximum(n_p, 1), np.nan)
    return pd.DataFrame({
        "bin": np.arange(n_bins), "avg_p_eagle": avg_p,
        "hit_rate": hit_rate, "n": n_p,
    })


calib_dfs = {}
for b in BENCHES:
    print(f"calibration scan: {b}")
    calib_dfs[b] = stream_anchor_calibration(b, n_bins=20)
list(calib_dfs.items())[0][1].head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
for b in BENCHES:
    df = calib_dfs[b].dropna()
    df = df[df["n"] > 1000]
    ax.plot(df["avg_p_eagle"], df["hit_rate"], marker="o",
            color=PALETTE[b], label=b)
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="perfect calib")
ax.set_xlabel("avg `anchor_path_prob_eagle` in bin")
ax.set_ylabel("observed `anchor_hit_rate`")
ax.set_title("EAGLE calibration")
ax.legend()
plt.tight_layout()
plt.show()

**Reading**: above the y=x line ⇒ EAGLE underestimates accept (conservative). Below ⇒ overconfident. Sharp deviation at high p_eagle ⇒ recalibration would help.

## P3.4 — Feature lift (which signals predict suffix acceptance?)

Pre-aggregated in `05_feature_bins_summary.tsv` — equal-frequency 10 bins, with `avg_suffix_accepted_len` and `top_bin_lift` per feature.

In [ ]:
fb_dfs = []
for b in BENCHES:
    p = DATA_ROOT / b / "05_feature_bins_summary.tsv"
    fb_dfs.append(load_tsv(p))
fb_df = pd.concat(fb_dfs, ignore_index=True)

# top bin lift = ratio of top-decile accept vs global mean.
lift_table = fb_df.groupby(["benchmark", "feature_name"]).agg(
    top_bin_lift=("top_bin_lift", "max"),
).reset_index()
lift_pivot = lift_table.pivot(index="feature_name", columns="benchmark",
                              values="top_bin_lift")
lift_pivot.style.format("{:.2f}")

In [ ]:
g = sns.relplot(
    fb_df,
    x="bin_id", y="avg_suffix_accepted_len",
    col="feature_name", hue="benchmark",
    kind="line", marker="o", palette=PALETTE,
    col_wrap=3, height=3.2, aspect=1.3,
    facet_kws={"sharey": False},
)
g.set_axis_labels("feature bin (low → high)", "avg suffix accepted len")
g.fig.suptitle("How feature value relates to suffix accept length", y=1.02)
plt.show()

**Reading**: features whose bin curve is monotonically rising have predictive signal. A high `top_bin_lift` (in the table) means *the top decile is N× the mean* — a high-precision selection rule.

## Summary panel

In [ ]:
rows = []
for b in BENCHES:
    s = step_df[step_df["benchmark"] == b]
    c = cost_df[cost_df["benchmark"] == b]
    bb_dep_mean = s["accepted_max_depth_backbone"].mean()
    ext_dep_mean = s["accepted_max_depth_extended"].mean()
    extra_acc = ext_dep_mean - bb_dep_mean
    extra_cost = (c["total_latency_max_tree_ms"] - c["total_latency_backbone_only_ms"]).mean()
    rows.append({
        "benchmark": b,
        "avg_acc_len_bb": bb_dep_mean,
        "avg_acc_len_ext": ext_dep_mean,
        "suffix_extra_acc": extra_acc,
        "suffix_extra_cost_ms": extra_cost,
        "cost_per_extra_acc": extra_cost / max(extra_acc, 1e-6),
    })
pd.DataFrame(rows).round(3)

**Interpretation of summary**: `cost_per_extra_acc` (ms / extra accepted token from suffix) is the headline number. Compare against the *target's per-token latency* (≈ `vanilla_step_ms` from `latency_config`, ~40 ms). If cost_per_extra_acc << 40, suffix max-tree is a net win; if comparable, selection is needed.